# Order Book Signals on Bond & Equity Calendar Spreads

**Goal:** Investigate whether order-book signals (OFI, size imbalance) on individual futures legs can predict short-horizon calendar spread returns during the roll period.

**Calendar spread** = price of the front (near) contract minus the back (far) contract. During the *roll period* (the ~4 weeks before front-contract expiry) both legs are actively traded and the spread is liquid.

**Three OFI signals per leg:**
- `ofi_vol` – signed volume fraction: buy-initiated vs sell-initiated trades
- `ofi_quote` – quote-level OFI: Δbid_size − Δask_size (passive-side pressure)
- `ofi_size` – static level-1 imbalance: (twa_bid_size − twa_ask_size) / total

**Calendar-spread signals:** near signal − far signal for each of the above.

---
**Sections**
1. Setup & Connection
2. Data Exploration
3. Fetch Intraday Data
4. Calendar Spread Construction
5. Order Book Signal Construction
6. Daily Aggregation
7. Signal Analysis (IC)
8. Visualisation

## 1  Setup & Connection

In [ ]:
import sys
sys.path.insert(0, '..')

import polars as pl
import polars.selectors as cs
import numpy as np
import scipy.stats as stats
from plotnine import *
from plotnine.themes import theme_bw

from src.utils import (
    connect_snowflake,
    create_snowpark_session,
    retrieve_polars_from_snowpark,
    unpack_kwargs,
    unpack_kwargs_for_agg,
    parse_expiry,
    rank_contracts,
    build_calendar_spread,
    add_mid_prices,
    add_ofi_signals,
    add_spread_signals,
    add_forward_cs_return,
    rank_ic,
)
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(20)
print('polars', pl.__version__)

In [ ]:
snowflake_conn = connect_snowflake('../.env')
snowpark = create_snowpark_session('../.env')
print('Connected.')

## 2  Data Exploration

First, see which `qcode` values are available and how much data exists for each.

In [ ]:
df_catalog = pl.read_database(
    """
    SELECT
        QCODE,
        COUNT(DISTINCT SECURITY)       AS n_contracts,
        COUNT(DISTINCT PUBLICATION_DATE) AS n_dates,
        MIN(PUBLICATION_DATE)          AS first_date,
        MAX(PUBLICATION_DATE)          AS last_date,
        COUNT(*)                       AS n_rows
    FROM LISTED_INTERN_PROJECT.PROJECT_1.BINNED_DATA
    GROUP BY QCODE
    ORDER BY n_rows DESC
    """,
    snowflake_conn,
).select(pl.all().name.to_lowercase())

df_catalog

In [ ]:
# Sample a few securities to confirm ticker format
pl.read_database(
    """
    SELECT DISTINCT QCODE, SECURITY
    FROM LISTED_INTERN_PROJECT.PROJECT_1.BINNED_DATA
    ORDER BY QCODE, SECURITY
    LIMIT 40
    """,
    snowflake_conn,
).select(pl.all().name.to_lowercase())

## 3  Fetch Intraday Data

**Update `LS_BOND_QCODES` and `LS_EQUITY_QCODES`** based on the catalog output above.

Common codes to look for:
- **Bond futures** – `TY` (10yr Treasury), `FV` (5yr), `TU` (2yr), `US` (30yr Bond), `RX` (Euro Bund), `DU` (Schatz), `OE` (Bobl), `JB` (JGB)
- **Equity futures** – `ES` (S&P 500), `NQ` (Nasdaq), `Z` (FTSE 100), `SX` (EuroStoxx 50), `NM` (Nikkei mini), `NK` (Nikkei)

In [ ]:
# ── Fill these in after reviewing df_catalog ──────────────────────────────────
LS_BOND_QCODES   = ['TY', 'FV', 'RX']   # bond futures
LS_EQUITY_QCODES = ['ES', 'NQ', 'SX']   # equity index futures
# ─────────────────────────────────────────────────────────────────────────────

ALL_QCODES = LS_BOND_QCODES + LS_EQUITY_QCODES
QCODES_SQL = ', '.join(f"'{q}'" for q in ALL_QCODES)

DATA_START_DATE = '2022-01-01'
DATA_END_DATE   = '2024-12-31'
ROLL_DAYS       = 20   # trading days before near-contract expiry to define roll window

print(f'Fetching qcodes: {ALL_QCODES}')
print(f'Date range: {DATA_START_DATE} → {DATA_END_DATE}')

In [ ]:
df_raw = pl.read_database(
    f"""
    SELECT
        QCODE,
        PUBLICATION_DATE  AS date,
        SECURITY,
        BIN_START_TIME,
        GMT_OFFSET_HOURS,
        OPEN, HIGH, LOW, CLOSE,
        VOLUME,
        SIGNED_VOLUME,
        TRADE_COUNT,
        TWA_BID_SIZE,
        TWA_ASK_SIZE,
        BID_SIZE_START,
        ASK_SIZE_START,
        BID_SIZE_END,
        ASK_SIZE_END,
        BID_START,
        BID_END,
        ASK_START,
        ASK_END,
        TWA_BID,
        TWA_ASK
    FROM LISTED_INTERN_PROJECT.PROJECT_1.BINNED_DATA
    WHERE
        QCODE IN ({QCODES_SQL})
        AND PUBLICATION_DATE >= '{DATA_START_DATE}'
        AND PUBLICATION_DATE <= '{DATA_END_DATE}'
    ORDER BY QCODE, SECURITY, PUBLICATION_DATE, BIN_START_TIME
    """,
    snowflake_conn,
).select(pl.all().name.to_lowercase())

print(df_raw.shape)
df_raw.head(5)

## 4  Calendar Spread Construction

Steps:
1. Parse expiry date from Bloomberg ticker (e.g. `TY2024H Govt` → 2024-03-01)
2. Rank contracts by expiry within each (qcode, date, bin) — rank 1 = front month
3. Join front and back rows into a single spread row
4. Filter to the roll period: `0 < days_to_expiry ≤ ROLL_DAYS`

In [ ]:
df = (
    df_raw
    .pipe(parse_expiry)
    .pipe(rank_contracts)
)

# Sanity check: confirm two contracts per (qcode, date, bin) during the roll
df.filter(
    pl.col('contract_rank') <= 2
).group_by(['qcode', 'date', 'bin_start_time']).agg(
    n_contracts=pl.col('security').n_unique()
).filter(pl.col('n_contracts') == 2).height

In [ ]:
df_spread = df.pipe(build_calendar_spread, roll_days=ROLL_DAYS)

print(f'Spread rows: {df_spread.shape[0]:,}')
df_spread.select(['qcode', 'date', 'bin_start_time', 'security', 'security_far',
                  'expiry_date', 'expiry_date_far', 'days_to_expiry']).head(8)

## 5  Order Book Signal Construction

Add mid prices and OFI signals for each leg, then compute calendar-spread-level signals.

In [ ]:
df_signals = (
    df_spread
    # mid prices for near and far legs
    .pipe(add_mid_prices, suffix='')
    .pipe(add_mid_prices, suffix='_far')
    # OFI signals for near and far legs
    .pipe(add_ofi_signals, suffix='')
    .pipe(add_ofi_signals, suffix='_far')
    # calendar spread signals (near − far)
    .pipe(add_spread_signals)
    # forward intraday return on the spread (next bin)
    .pipe(add_forward_cs_return, n_periods=1)
)

signal_cols = ['qcode', 'date', 'bin_start_time', 'security', 'security_far',
               'cs_mid', 'cs_return_fwd',
               'ofi_vol', 'ofi_quote', 'ofi_size',
               'cs_ofi_vol', 'cs_ofi_quote', 'cs_ofi_size']
df_signals.select(signal_cols).head(8)

In [ ]:
# Signal descriptive stats
df_signals.select(
    ['ofi_vol', 'ofi_quote', 'ofi_size',
     'cs_ofi_vol', 'cs_ofi_quote', 'cs_ofi_size', 'cs_return_fwd']
).describe()

## 6  Daily Aggregation

Aggregate intraday bins to daily-level signals and next-day spread returns.
Daily OFI = sum(signed_volume) / sum(volume); daily close = last bin mid.

In [ ]:
df_daily = (
    df_signals
    .group_by(['qcode', 'date', 'security', 'security_far'])
    .agg(
        # daily OFI = volume-weighted average of per-bin signed_volume fraction
        ofi_vol_d      = (pl.col('signed_volume').sum() /
                          (pl.col('volume').sum() + 1e-10)),
        ofi_vol_d_far  = (pl.col('signed_volume_far').sum() /
                          (pl.col('volume_far').sum() + 1e-10)),
        # daily quote OFI = mean of intraday values
        ofi_quote_d    = pl.col('ofi_quote').mean(),
        ofi_quote_d_far= pl.col('ofi_quote_far').mean(),
        # daily size imbalance = mean of intraday values
        ofi_size_d     = pl.col('ofi_size').mean(),
        ofi_size_d_far = pl.col('ofi_size_far').mean(),
        # calendar spread close (last bin)
        cs_mid_close   = pl.col('cs_mid').last(),
    )
    .sort(['qcode', 'security', 'security_far', 'date'])
    .with_columns(
        cs_ofi_vol_d   = pl.col('ofi_vol_d')   - pl.col('ofi_vol_d_far'),
        cs_ofi_quote_d = pl.col('ofi_quote_d') - pl.col('ofi_quote_d_far'),
        cs_ofi_size_d  = pl.col('ofi_size_d')  - pl.col('ofi_size_d_far'),
    )
    # next-day return on the calendar spread
    .with_columns(
        cs_mid_next = pl.col('cs_mid_close').shift(-1).over(
            ['qcode', 'security', 'security_far'],
            order_by='date',
        )
    )
    .with_columns(
        cs_return_1d = (
            (pl.col('cs_mid_next') - pl.col('cs_mid_close')) /
            pl.col('cs_mid_close').abs().clip(lower_bound=1e-10)
        )
    )
    .drop_nulls(subset=['cs_return_1d'])
)

print(df_daily.shape)
df_daily.head(5)

## 7  Signal Analysis — Information Coefficient (IC)

IC = Spearman rank correlation between signal_t and forward return_{t+1}.
- IC > 0: signal predicts positive return
- |IC| > 0.05 is typically considered meaningful in practice
- t-stat = IC × √N / √(1 − IC²)

In [ ]:
SIGNAL_COLS = ['ofi_vol_d', 'ofi_quote_d', 'ofi_size_d',
               'cs_ofi_vol_d', 'cs_ofi_quote_d', 'cs_ofi_size_d']
TARGET_COL  = 'cs_return_1d'

ASSET_CLASS_MAP = {q: 'Bond' for q in LS_BOND_QCODES}
ASSET_CLASS_MAP.update({q: 'Equity' for q in LS_EQUITY_QCODES})

rows = []
for qcode, grp in df_daily.group_by('qcode'):
    qcode = qcode[0]
    for sig in SIGNAL_COLS:
        ic_val = rank_ic(grp[sig], grp[TARGET_COL])
        n      = grp.drop_nulls(subset=[sig, TARGET_COL]).height
        t_stat = ic_val * (n ** 0.5) / max((1 - ic_val**2) ** 0.5, 1e-10) if not np.isnan(ic_val) else float('nan')
        rows.append({
            'qcode': qcode,
            'asset_class': ASSET_CLASS_MAP.get(qcode, 'Unknown'),
            'signal': sig,
            'ic': round(ic_val, 4),
            't_stat': round(t_stat, 2),
            'n_obs': n,
        })

df_ic = pl.DataFrame(rows).sort(['asset_class', 'qcode', 'signal'])
df_ic

In [ ]:
# Mean IC by signal and asset class
df_ic.group_by(['asset_class', 'signal']).agg(
    mean_ic   = pl.col('ic').mean().round(4),
    mean_tstat= pl.col('t_stat').mean().round(2),
).sort(['asset_class', 'signal'])

In [ ]:
# Rolling IC (21-day window) for the best intraday signal
BEST_SIGNAL = 'cs_ofi_vol_d'   # update after inspecting df_ic

df_rolling_ic = []
for qcode, grp in df_daily.sort('date').group_by('qcode'):
    qcode = qcode[0]
    dates = grp['date'].to_list()
    sig   = grp[BEST_SIGNAL].to_list()
    ret   = grp[TARGET_COL].to_list()
    W = 21
    for i in range(W - 1, len(dates)):
        s_w = pl.Series(sig[i-W+1:i+1])
        r_w = pl.Series(ret[i-W+1:i+1])
        ic_val = rank_ic(s_w, r_w)
        df_rolling_ic.append({'qcode': qcode, 'date': dates[i], 'rolling_ic': ic_val})

df_rolling_ic = pl.DataFrame(df_rolling_ic)
df_rolling_ic.head(5)

## 8  Visualisation

In [ ]:
# 8a: Calendar spread price over time (one pair per qcode)

# Take the pair with the most roll-period observations per qcode
top_pair = (
    df_daily
    .group_by(['qcode', 'security', 'security_far'])
    .agg(n=pl.len())
    .sort('n', descending=True)
    .group_by('qcode')
    .first()
    .select(['qcode', 'security', 'security_far'])
)

df_cs_plot = (
    df_daily
    .join(top_pair, on=['qcode', 'security', 'security_far'], how='inner')
    .with_columns(
        asset_class=pl.col('qcode').map_elements(
            lambda q: ASSET_CLASS_MAP.get(q, 'Unknown'), return_dtype=pl.Utf8
        )
    )
    .to_pandas()
)

(
    ggplot(df_cs_plot, aes(x='date', y='cs_mid_close', colour='qcode'))
    + geom_line(size=0.6)
    + facet_wrap('~asset_class + qcode', scales='free_y', ncol=2)
    + labs(
        title='Calendar Spread Mid-Price During Roll Period',
        x='Date', y='Spread (near − far)',
    )
    + theme_bw()
    + theme(legend_position='none', figure_size=(12, 8),
            axis_text_x=element_text(angle=45, hjust=1))
)

In [ ]:
# 8b: OFI signal distributions by asset class

df_ofi_plot = (
    df_daily
    .with_columns(
        asset_class=pl.col('qcode').map_elements(
            lambda q: ASSET_CLASS_MAP.get(q, 'Unknown'), return_dtype=pl.Utf8
        )
    )
    .select(['qcode', 'asset_class', 'cs_ofi_vol_d', 'cs_ofi_quote_d', 'cs_ofi_size_d'])
    .unpivot(
        index=['qcode', 'asset_class'],
        on=['cs_ofi_vol_d', 'cs_ofi_quote_d', 'cs_ofi_size_d'],
        variable_name='signal',
        value_name='value',
    )
    .filter(pl.col('value').is_not_nan())
    # clip extreme values for display
    .with_columns(value=pl.col('value').clip(-5, 5))
    .to_pandas()
)

(
    ggplot(df_ofi_plot, aes(x='asset_class', y='value', fill='asset_class'))
    + geom_violin(alpha=0.6, draw_quantiles=[0.25, 0.5, 0.75])
    + facet_wrap('~signal', ncol=3)
    + labs(
        title='Calendar Spread OFI Signal Distributions',
        x='Asset Class', y='Signal Value',
    )
    + theme_bw()
    + theme(legend_position='none', figure_size=(12, 5))
)

In [ ]:
# 8c: IC bar chart across signals and qcodes

df_ic_plot = df_ic.to_pandas()

(
    ggplot(df_ic_plot, aes(x='signal', y='ic', fill='asset_class'))
    + geom_col(position='dodge')
    + geom_hline(yintercept=0, linetype='dashed', colour='black')
    + facet_wrap('~qcode', ncol=3)
    + labs(
        title='IC by Signal and Futures Contract',
        x='Signal', y='Rank IC (Spearman)',
    )
    + theme_bw()
    + theme(axis_text_x=element_text(angle=45, hjust=1),
            figure_size=(14, 8))
)

In [ ]:
# 8d: Rolling IC over time (21-day window)

df_ric_plot = (
    df_rolling_ic
    .with_columns(
        asset_class=pl.col('qcode').map_elements(
            lambda q: ASSET_CLASS_MAP.get(q, 'Unknown'), return_dtype=pl.Utf8
        )
    )
    .filter(pl.col('rolling_ic').is_not_nan())
    .to_pandas()
)

(
    ggplot(df_ric_plot, aes(x='date', y='rolling_ic', colour='qcode'))
    + geom_line(alpha=0.8)
    + geom_hline(yintercept=0, linetype='dashed', colour='black')
    + facet_wrap('~asset_class', ncol=1, scales='free_x')
    + labs(
        title=f'Rolling 21-Day IC: {BEST_SIGNAL}',
        x='Date', y='Rolling IC',
    )
    + theme_bw()
    + theme(figure_size=(12, 8),
            axis_text_x=element_text(angle=45, hjust=1))
)

In [ ]:
# 8e: Signal vs forward return scatter with regression line (one qcode)

FOCUS_QCODE = LS_BOND_QCODES[0]   # change as needed

df_scatter = (
    df_daily
    .filter(pl.col('qcode') == FOCUS_QCODE)
    .select([BEST_SIGNAL, TARGET_COL])
    .drop_nulls()
    # clip for display
    .filter(
        pl.col(BEST_SIGNAL).is_between(-3, 3) &
        pl.col(TARGET_COL).is_between(-0.05, 0.05)
    )
    .to_pandas()
)

(
    ggplot(df_scatter, aes(x=BEST_SIGNAL, y=TARGET_COL))
    + geom_point(alpha=0.2, size=0.8)
    + geom_smooth(method='lm', colour='red', fill='lightcoral')
    + labs(
        title=f'{FOCUS_QCODE}: {BEST_SIGNAL} vs Next-Day CS Return',
        x=BEST_SIGNAL, y='Next-Day Calendar Spread Return',
    )
    + theme_bw()
    + theme(figure_size=(8, 5))
)